# Public Dialogue Analyser

## Jessica K. Montgomery

## May 12th 2026

## Purpose

This notebook analyses 65 UK public dialogue documents on science and technology to identify:

- **Shared concerns or hopes** that recur across different technologies
- **Concerns or benefits that are distinctive to public dialogues on artificial intelligence (AI)** compared to other technologies
- **How public dialogue about AI changes over time**


## Corpus

The corpus consists of UK public dialogue reports on science and technology, spanning multiple technologies (including AI) and multiple years. Each document is associated with metadata including technology domain(s) and publication year.

PDFs are ingested directly and converted to paragraph-level text for analysis.



In [ ]:
# @title Install pub-dialogue package
# Local: run 'pip install -e .' once in the repo root.
# Colab:  installs directly from GitHub.
try:
    import pub_dialogue
except ImportError:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/mlatcl/pub-dialogue.git",
    ])
    import pub_dialogue

This code cell imports essential libraries and modules required for processing public dialogue documents, including tools for data manipulation, natural language processing, and clustering analysis. By setting up the environment with libraries like PyMuPDF for PDF handling and scikit-learn for clustering, it lays the groundwork for the subsequent extraction of concern and benefit phrases, semantic embedding, and analysis through k-means clustering, which are crucial for understanding public sentiment in the context of the UK dialogue documents.

This cell sets up the necessary libraries and configurations for the analysis, including random number generation, numerical operations, and plotting capabilities. By establishing a consistent plotting style and resolution, it ensures that visualizations of the extracted concern phrases and their clustering will be clear and aesthetically pleasing, which is crucial for interpreting the results effectively in the context of public dialogue research.

In [ ]:
# @title Import libraries and define configuration

import random
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150


This cell establishes the foundational setup for processing public dialogue documents by ensuring reproducibility through a fixed random seed and configuring model parameters for embedding and clustering. It also addresses potential issues with text chunking from PDFs, particularly when paragraph separations are inadequate, by implementing a fallback mechanism to ensure effective extraction of concern phrases. This setup is crucial for maintaining consistency across analyses and enhancing the robustness of the subsequent extraction and clustering processes, ultimately contributing to a more reliable understanding of public sentiments.

In [ ]:
# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Model Configuration
EMBEDDING_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-4o-mini"

# Clustering - now 75 clusters for concern phrases (smaller units than paragraphs)
N_CONCERN_CLUSTERS = 75
SOFT_MEMBERSHIP_THRESHOLD = 0.3

# Chunking
MIN_CHUNK_WORDS = 40
MAX_CHUNK_WORDS = 500
MIN_CHUNK_CHARS = 80

# When paragraph-level splitting (the default) fails to produce reasonable
# chunks (because the PDF lacks double-newline paragraph separators),
# fall back to sentence-level splitting and repack into chunks of about
# this many words. v18: addresses an issue identified in v17 where 20
# documents (mostly AI policy reports from 2020+) were silently truncated
# because their full text was treated as a single 500-word paragraph.
SENTENCE_FALLBACK_TARGET_WORDS = 300
SENTENCE_FALLBACK_MIN_PARAGRAPHS = 3  # if Method 1 yields fewer paragraphs, fall back

# Processing
EMBEDDING_BATCH_SIZE = 100
EXTRACTION_BATCH_SIZE = 10  # For concern extraction

# Paths — resolve sensibly for both Colab (/content/...) and local (repo root)
try:
    import google.colab  # noqa: F401
    _REPO_ROOT = Path("/content")
except ImportError:
    # Local: notebook lives in repo root, so relative paths work directly
    _REPO_ROOT = Path(".")

PDF_FOLDER        = _REPO_ROOT / "pdfs"
OUTPUT_FOLDER     = _REPO_ROOT / "outputs"
CHECKPOINT_FOLDER = _REPO_ROOT / "checkpoints"
DATA_FOLDER       = _REPO_ROOT / "data"

# Ensure output directories exist
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
CHECKPOINT_FOLDER.mkdir(parents=True, exist_ok=True)

This cell initializes the necessary directories for processing public dialogue documents, ensuring that the required folders for PDF ingestion, output storage, and checkpoints are created if they do not already exist. Additionally, it suppresses warnings to maintain a clean output and confirms the configuration settings for the embedding model, target clusters, and language model used for extracting concern phrases, which are crucial for the subsequent analysis and clustering steps in the pipeline.

In [ ]:
for folder in [PDF_FOLDER, OUTPUT_FOLDER, CHECKPOINT_FOLDER]:
    folder.mkdir(exist_ok=True)


import warnings
warnings.filterwarnings('ignore')

print("Configuration loaded")
print(f"  Embedding model: {EMBEDDING_MODEL}")
print(f"  Target clusters: {N_CONCERN_CLUSTERS}")
print(f"  LLM for extraction: {LLM_MODEL}")

This cell imports essential modules and functions from the `pub_dialogue` library, which are critical for processing and analyzing public dialogue data. By establishing access to shared utilities for phrase extraction, semantic embedding, and clustering, it sets the foundation for subsequent analysis steps, ensuring that the pipeline can efficiently handle tasks such as data validation and diagnostics, which are vital for maintaining the integrity and accuracy of the overall analysis.

In [ ]:
# @title Load pub_dialogue — shared helper functions and access/assess/address modules
try:
    import pub_dialogue.utils as du
    import pub_dialogue.access as access
    import pub_dialogue.assess as assess
    import pub_dialogue.address as address
    from pub_dialogue.utils import (
        show_status, show_complete, show_warning,
        save_checkpoint, load_checkpoint,
        CROSSCUTTING_ENTROPY_THRESHOLD,
        EXTRACTION_PROMPT, BENEFIT_EXTRACTION_PROMPT,
        ExtractionResult, extract_phrases, validate_extraction_cache,
        write_extraction_diagnostics,
        filter_missing_source_text, get_embeddings_batch,
        label_cluster, pretty_label, clusters_to_labels,
        clusters_to_lenses, html_escape,
        normalized_entropy, hhi, topk_share, parse_year, tokenize,
        is_privacy_text, entropy_by_year, ai_fingerprint_over_crosscut,
        run_sensitivity,
        vocabulary_frequency_diagnostic, generate_validation_summary,
        extract_chunks_from_pdf, reset_chunk_stats, get_chunk_stats,
        MIN_CHUNK_WORDS, MIN_CHUNK_CHARS, MAX_CHUNK_WORDS,
        SENTENCE_FALLBACK_TARGET_WORDS, SENTENCE_FALLBACK_MIN_PARAGRAPHS,
    )
    print("pub_dialogue imported successfully")
except ImportError as _e:
    print(f"pub_dialogue not found: {_e}")
    raise

This cell sets up the necessary API access for the language model and embedding model used in the analysis. By securely loading the appropriate API keys, either from environment variables or user input, it ensures that the subsequent steps in the pipeline can effectively utilize the specified models for processing public dialogue documents. This configuration is crucial as it establishes the foundation for the extraction and clustering tasks that follow, directly impacting the quality and accuracy of the analysis.

In [ ]:
from pub_dialogue.client import LLMClient


In [ ]:
# @title Configure API access
import os as _os

# Load .env file if present (local development)
try:
    from dotenv import load_dotenv as _load_dotenv
    _load_dotenv()
except ImportError:
    pass

# Infer which API key env-var to prompt for from the model name prefix
_key_map = {"claude": "ANTHROPIC_API_KEY", "gemini": "GOOGLE_API_KEY"}
_key_var = next(
    (v for k, v in _key_map.items() if LLM_MODEL.startswith(k)),
    "OPENAI_API_KEY",
)

# Try Colab secrets first
try:
    from google.colab import userdata as _userdata
    _secret = _userdata.get(_key_var)
    if _secret:
        _os.environ[_key_var] = _secret
        print(f"{_key_var} loaded from Colab secrets")
except Exception:
    pass

# Interactive prompt fallback
if not _os.environ.get(_key_var):
    from getpass import getpass as _getpass
    _os.environ[_key_var] = _getpass(f"Enter {_key_var}: ")
    print(f"{_key_var} entered manually")

client = LLMClient(model=LLM_MODEL, embedding_model=EMBEDDING_MODEL)
print(f"LLMClient ready — model: {LLM_MODEL}, embeddings: {EMBEDDING_MODEL}")

# Suppress litellm's noisy stderr; pub_dialogue.address uses Python logging instead
import logging as _logging
_logging.getLogger("LiteLLM").setLevel(_logging.ERROR)
_logging.getLogger("pub_dialogue").setLevel(_logging.WARNING)

# Proactive rate limiter: stay under the 500 RPM hard limit with 10% headroom.
# This throttles before each request so workers don't burst through the limit
# simultaneously, complementing the per-request exponential backoff retry.
du.configure_rate_limiter(450)

---
# SECTION 1: Data ingestion and preprocessing
---

In [ ]:
# @title Upload source PDF documents
show_status("Preparing PDF upload...")

try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

pdf_files = []

if IN_COLAB:
    print("Upload your PDF files:")
    uploaded = _colab_files.upload()
    for filename, content in uploaded.items():
        if filename.endswith('.pdf'):
            filepath = PDF_FOLDER / filename
            filepath.write_bytes(content)
            pdf_files.append(filepath)
else:
    # Local execution: scan DATA_FOLDER for existing PDFs
    DATA_FOLDER.mkdir(parents=True, exist_ok=True)
    pdf_files = sorted(DATA_FOLDER.glob("*.pdf"))
    if not pdf_files:
        raise FileNotFoundError(
            f"No PDF files found in {DATA_FOLDER.resolve()}. "
            "Place your corpus PDFs there before running this cell."
        )
    print(f"Found {len(pdf_files)} PDFs in {DATA_FOLDER}/")

show_complete(f"Loaded {len(pdf_files)} PDF files")

This cell facilitates the ingestion of document metadata, which is crucial for understanding the context of the subsequent analysis. By loading an Excel file containing information about each document's filename, technology, and year, it ensures that the analysis can accurately reference and categorize the public dialogue documents, thereby enhancing the interpretability of the extracted concern and benefit phrases in the overall pipeline. The validation checks for required columns further ensure data integrity, which is vital for the reliability of the subsequent NLP tasks.

In [ ]:
import pandas as pd


In [ ]:
# @title Upload document metadata
show_status("Upload metadata file...")

metadata_df = None

if IN_COLAB:
    print("Upload Excel file with columns: filename, technology, year")
    uploaded = _colab_files.upload()
    for fn, content in uploaded.items():
        if fn.endswith(('.xlsx', '.xls')):
            path = OUTPUT_FOLDER / fn
            path.write_bytes(content)
            metadata_df = pd.read_excel(path)
            break
    if metadata_df is None:
        raise ValueError("No Excel file uploaded!")
else:
    # Local execution: look for an xlsx in OUTPUT_FOLDER then repo root
    _candidates = list(DATA_FOLDER.glob("*.xlsx")) + list(Path(".").glob("*.xlsx"))
    if not _candidates:
        raise FileNotFoundError(
            "No metadata xlsx found. Place tech_metadata.xlsx in outputs/ or repo root."
        )
    metadata_df = pd.read_excel(_candidates[0])
    print(f"Loaded metadata from {_candidates[0]}")

required = ['filename', 'technology', 'year']
missing = [c for c in required if c not in metadata_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

metadata_lookup = metadata_df.set_index('filename').to_dict('index')

print(f"\nTechnology Distribution:")
print(metadata_df['technology'].value_counts().to_string())
print(f"\nYear Range: {metadata_df['year'].min()} - {metadata_df['year'].max()}")

show_complete(f"Metadata loaded for {len(metadata_df)} documents")

This cell processes PDF documents to extract paragraph-level text, employing a hybrid chunking strategy that accommodates varying document structures. By categorizing the extracted text into distinct methods based on the presence of paragraph breaks and internal sentence splits, it ensures that the subsequent analysis can leverage well-structured data, which is crucial for accurately identifying concerns and benefits in public dialogue. The resulting chunks are then summarized and saved for further semantic analysis, laying the groundwork for effective clustering and insight generation in the overall pipeline.

In [ ]:
from tqdm.notebook import tqdm


In [ ]:
# @title Extract paragraph-level text from PDFs (v19 three-case hybrid chunker)
# Helper functions (_split_into_sentences, _repack_sentences_into_chunks,
# _paragraph_split, extract_chunks_from_pdf) are provided by dialogue_utils.
# The three-case strategy:
#   Case 1 — paragraph-only (most documents)
#   Case 2 — paragraphs + internal sentence-split of oversized paragraphs
#   Case 3 — full sentence-fallback (PDFs lacking paragraph breaks)

show_status("Extracting paragraph chunks...")

reset_chunk_stats()

all_chunks = []
for pdf_path in tqdm(pdf_files, desc="Processing PDFs"):
    if pdf_path.name not in metadata_lookup:
        raise ValueError(f"No metadata found for PDF: {pdf_path.name}")
    metadata = metadata_lookup[pdf_path.name]
    chunks = extract_chunks_from_pdf(pdf_path, metadata)
    all_chunks.extend(chunks)

chunks_df = pd.DataFrame(all_chunks)
chunks_df["chunk_id"] = [f"chunk_{i}" for i in range(len(chunks_df))]

_chunk_stats = get_chunk_stats()

print(f"\nExtraction Summary:")
print(f"  Documents:                                     {chunks_df['source_file'].nunique()}")
print(f"  Documents w/ pure paragraph segmentation:      {_chunk_stats['documents_paragraph_only']}")
print(f"  Documents w/ paragraphs + some sentence-split: {_chunk_stats['documents_paragraph_with_split']}")
print(f"  Documents w/ full sentence-fallback:           {_chunk_stats['documents_sentence_fallback']}")
print(f"  Oversized paragraphs split internally:         {_chunk_stats['oversized_paragraphs_split']}")
print(f"  Chunks produced by sentence-split:             {_chunk_stats['chunks_from_sentence_split']}")
print(f"  Chunks produced by full sentence-fallback:     {_chunk_stats['chunks_from_sentence_fallback']}")
print(f"  Total chunks (paragraph-derived):              {(chunks_df['chunking_method']=='paragraph').sum()}")
print(f"  Paragraphs seen (raw):                         {_chunk_stats['paragraphs_seen']}")
print(f"  Below {MIN_CHUNK_WORDS}-word floor:                          {_chunk_stats['paragraphs_below_word_floor']}")
print(f"  Below {MIN_CHUNK_CHARS}-char floor:                          {_chunk_stats['paragraphs_below_char_floor']}")
print(f"  Truncated (safety net):                        {_chunk_stats['paragraphs_truncated']}")
print(f"  Kept after filter:                             {_chunk_stats['paragraphs_kept']}")
print(f"  Words per kept chunk (mean):                   {chunks_df['word_count'].mean():.0f}")

# Per-document summary
doc_summary = (chunks_df.groupby("source_file")
               .agg(paragraphs_kept=("chunk_id", "size"),
                    methods_used=("chunking_method", lambda s: "+".join(sorted(s.unique()))),
                    truncated_chunks=("was_truncated", "sum"))
               .reset_index()
               .sort_values("paragraphs_kept"))
doc_summary.to_csv(OUTPUT_FOLDER / "paragraph_chunks_per_document.csv", index=False)
print(f"\nDocuments using full sentence-fallback (PDF lacks paragraph breaks):")
fallback_docs = doc_summary[doc_summary["methods_used"] == "sentence_fallback"]
print(fallback_docs.to_string(index=False) if len(fallback_docs) else "  (none)")

chunks_df.to_csv(OUTPUT_FOLDER / "paragraph_chunks.csv", index=False)
show_complete(f"Extracted {len(chunks_df)} chunks from {chunks_df['source_file'].nunique()} documents")

TECHNOLOGY_CATEGORIES = sorted(metadata_df["technology"].dropna().unique().tolist())

# Sanity check: technology labels remain canonical
observed = set(chunks_df["technology_meta"].unique().tolist())
expected = set(TECHNOLOGY_CATEGORIES)
if observed != expected:
    print("\nWARNING: Technology labels in extracted chunks differ from metadata!")
    print("Observed-only:", sorted(observed - expected))
    print("Metadata-only:", sorted(expected - observed))

## Summary Paragraph Statistics

This cell evaluates and visualizes the quality of paragraph-level data extracted from the ingested documents, providing insights into potential issues such as inconsistencies or missing information. Understanding data quality is crucial for ensuring the reliability of subsequent analyses, particularly in extracting meaningful concern and benefit phrases, as it directly impacts the effectiveness of the LLM and the accuracy of the clustering results.

In [ ]:
# @title Summarise paragraph-level data quality
assess.plot_data_quality(chunks_df, OUTPUT_FOLDER)
plt.show()

## Chunk Quality Diagnostic

This cell evaluates the quality of text chunks extracted from the ingested documents by identifying those that may be contaminated with irrelevant content, such as bibliography entries or survey table rows. By reporting the proportion of flagged chunks based on specified minimum word and character thresholds, it provides crucial insights into the integrity of the data, ensuring that subsequent analyses on concern and benefit phrase extraction are based on high-quality, relevant text. This step is essential for maintaining the reliability of the overall analysis pipeline.

In [ ]:
show_status("Running chunk content-quality diagnostic...")

# Reports the proportion of kept chunks that look like bibliography fragments
# or survey-table rows, so the analyst can see the contamination rate at the
# chosen MIN_CHUNK_WORDS floor.
chunks_df = assess.flag_chunk_quality(chunks_df, OUTPUT_FOLDER, MIN_CHUNK_WORDS, MIN_CHUNK_CHARS)
show_complete("Chunk quality diagnostic complete — see chunk_quality_flagged.csv")

# Concern extraction

Next we extract decontextualised public concern phrases from paragraph-level dialogue text. These concern phrases form the primary analytic units for all subsequent embedding, clustering, and comparative analysis.


In [ ]:
import json


In [ ]:
# @title Extract decontextualised concern phrases

from concurrent.futures import ThreadPoolExecutor, as_completed

show_status("Extracting decontextualised concern phrases...")

concerns_cache_file = CHECKPOINT_FOLDER / "extracted_concerns.json"
concerns_errors_file = OUTPUT_FOLDER / "extraction_errors_concern.csv"
results = []

# Load cache if it exists and matches current chunk set
all_concerns = None
retry_ids = set()
if concerns_cache_file.exists():
    print("Found cached concern extractions...")
    with open(concerns_cache_file) as f:
        cached_concerns = json.load(f)
    if set(cached_concerns.keys()) == set(chunks_df["chunk_id"].tolist()):
        all_concerns = cached_concerns
        # Identify previously-failed chunks so we can re-run only those
        _err_df = du.read_csv_safe(concerns_errors_file)
        if not _err_df.empty:
            _failed_ids = set(_err_df["chunk_id"].tolist())
            retry_ids = {k for k in _failed_ids
                         if k in all_concerns and len(all_concerns[k]["concerns"]) == 0}
        if retry_ids:
            show_status(f"Re-extracting {len(retry_ids)} previously-failed chunks...")
        else:
            print(f"Using cached extractions for {len(all_concerns)} paragraphs")
    else:
        print("Cache mismatch — will re-extract all chunks")
        all_concerns = None

# Determine which chunks to run
if all_concerns is None:
    all_concerns = {}
    chunks_to_run = list(chunks_df.iterrows())
elif retry_ids:
    chunks_to_run = [(i, row) for i, row in chunks_df.iterrows()
                     if row["chunk_id"] in retry_ids]
else:
    chunks_to_run = []

if chunks_to_run:
    chunk_metadata = chunks_df.set_index("chunk_id")[["technology", "year", "source_file"]].to_dict("index")

    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {
            executor.submit(du.extract_phrases, row, "concern", client): row[1]["chunk_id"]
            for row in chunks_to_run
        }
        for future in tqdm(as_completed(futures), total=len(futures), desc="Extracting concerns"):
            result = future.result()
            results.append(result)
            meta = chunk_metadata[result.chunk_id]
            all_concerns[result.chunk_id] = {
                "concerns": result.retained_phrases,
                "technology": meta["technology"],
                "year": int(meta["year"]) if pd.notna(meta["year"]) else None,
                "source_file": meta["source_file"],
            }

    with open(concerns_cache_file, "w") as f:
        json.dump(all_concerns, f, indent=2)

    du.write_extraction_diagnostics(results, "concern", OUTPUT_FOLDER)
    failures = [r for r in results if r.error]
    failure_rate = len(failures) / max(1, len(chunks_to_run))
    if failure_rate > 0.05:
        show_warning(f"Concern extraction failure rate ({failure_rate:.1%}) exceeds 5% — check extraction_errors_concern.csv")
    elif failures:
        show_warning(f"{len(failures)} concern extraction failures ({failure_rate:.1%}). See extraction_errors_concern.csv")

    show_complete(f"Extracted/updated concerns for {len(chunks_to_run)} paragraphs")

# Flatten to individual concern rows
concern_rows = [
    {
        "chunk_id": chunk_id,
        "concern": concern,
        "technology": data["technology"],
        "year": data["year"],
        "source_file": data["source_file"],
    }
    for chunk_id, data in all_concerns.items()
    for concern in data["concerns"]
]

_concern_cols = ["chunk_id", "concern", "technology", "year", "source_file"]
concerns_df = pd.DataFrame(concern_rows, columns=_concern_cols) if concern_rows else pd.DataFrame(columns=_concern_cols)
concerns_df["concern_id"] = [f"concern_{i}" for i in range(len(concerns_df))]

print(f"\nExtraction Summary:")
print(f"  Paragraphs in cache: {len(all_concerns)}")
print(f"  Total concern phrases: {len(concerns_df)}")
print(f"  Avg concerns per paragraph: {len(concerns_df) / max(1, len(all_concerns)):.2f}")
print(f"  Paragraphs with no concerns: {sum(1 for d in all_concerns.values() if len(d['concerns']) == 0)}")

concerns_df.to_csv(OUTPUT_FOLDER / "extracted_concerns.csv", index=False)

# Retry any chunks that still failed after the extraction loop above.
# Runs in-process so all retry log lines appear in this cell's output.
_concern_errors = OUTPUT_FOLDER / "extraction_errors_concern.csv"
_has_concern_errors = not du.read_csv_safe(_concern_errors).empty
if _has_concern_errors:
    import importlib.util as _ilu
    _spec = _ilu.spec_from_file_location(
        "retry_failed_extractions",
        _REPO_ROOT / "scripts" / "retry_failed_extractions.py",
    )
    _retry_mod = _ilu.module_from_spec(_spec)
    _spec.loader.exec_module(_retry_mod)
    _retry_mod._retry_kind("concern", chunks_df, client, dry_run=False, max_workers=5)
    concerns_df = pd.read_csv(OUTPUT_FOLDER / "extracted_concerns.csv")
    print(f"Post-retry: {len(concerns_df)} total concern rows")

Merge the concern phrases to their respective technologies to better contextualize public sentiments and opinions, facilitating a more nuanced understanding of how different technologies are perceived in the dialogue documents. This step is crucial for subsequent clustering and interpretation of the concerns in relation to specific technological themes.

In [ ]:
# Make sure concerns_df knows the technology of each paragraph
concerns_df = concerns_df.merge(
    chunks_df[["chunk_id", "technology_meta"]],
    on="chunk_id",
    how="left"
)


This cell inspects and displays a sample of extracted concern phrases categorized by technology, ensuring that the specific language related to each technology has been appropriately filtered out. This step is crucial for validating the effectiveness of the extraction process, as it confirms that the analysis focuses on general concerns rather than technology-specific jargon, thereby enhancing the interpretability and relevance of the findings in the context of public dialogue.

In [ ]:
# @title Inspect sample concern extractions

print("Sample extracted concerns by technology:")
print("(Checking that technology-specific language has been removed)\n")

for tech in concerns_df['technology_meta'].unique()[:6]:
    tech_concerns = concerns_df[concerns_df['technology_meta'] == tech]['concern'].head(8).tolist()
    print(f"\n{tech}:")
    for c in tech_concerns:
        print(f"  • {c}")

---
# SECTION 3: Embedding and clustering
---

In [ ]:
import time


In [ ]:
# @title Generate semantic embeddings for concern phrases

embeddings_file = CHECKPOINT_FOLDER / "concern_embeddings.npy"
concern_ids_file = CHECKPOINT_FOLDER / "concern_ids.json"

if embeddings_file.exists() and concern_ids_file.exists():
    print("Found cached embeddings...")
    embeddings = np.load(embeddings_file)
    with open(concern_ids_file) as f:
        cached_concern_ids = json.load(f)
    if cached_concern_ids == concerns_df["concern_id"].tolist():
        print(f"Using cached embeddings: {embeddings.shape}")
    else:
        print("Cache mismatch - regenerating")
        embeddings = None
else:
    embeddings = None

if embeddings is None:
    show_status(f"Generating embeddings for {len(concerns_df)} concern phrases...")
    texts = concerns_df["concern"].tolist()
    all_embeddings = []
    failed_batch_starts = []

    for i in tqdm(range(0, len(texts), EMBEDDING_BATCH_SIZE), desc="Embedding"):
        batch = texts[i:i + EMBEDDING_BATCH_SIZE]
        try:
            all_embeddings.append(du.get_embeddings_batch(batch, client))
        except Exception as e:
            print(f"Error on batch {i}: {e}")
            failed_batch_starts.append(i)
        time.sleep(0.1)

    if failed_batch_starts:
        raise RuntimeError(
            f"{len(failed_batch_starts)} embedding batch(es) failed: "
            f"starting indices {failed_batch_starts}. Re-run to regenerate."
        )

    embeddings = np.vstack(all_embeddings)
    np.save(embeddings_file, embeddings)
    with open(concern_ids_file, "w") as f:
        json.dump(concerns_df["concern_id"].tolist(), f)
    show_complete(f"Generated embeddings: {embeddings.shape}")

assert embeddings.shape[0] == len(concerns_df), (
    f"Embeddings count ({embeddings.shape[0]}) does not match concerns_df rows ({len(concerns_df)}). "
    f"Delete {embeddings_file} and re-run."
)
print(f"Embedding dimensions: {embeddings.shape[1]}")

---
# SECTION 3B: Embedding and clustering (benefits)
---

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
# @title Extract decontextualised benefit phrases



show_status("Extracting decontextualised benefit phrases...")

benefits_cache_file = CHECKPOINT_FOLDER / "extracted_benefits.json"
benefits_errors_file = OUTPUT_FOLDER / "extraction_errors_benefit.csv"
results = []

# Load cache if it exists and matches current chunk set
all_benefits = None
retry_ids = set()
if benefits_cache_file.exists():
    print("Found cached benefit extractions...")
    with open(benefits_cache_file) as f:
        cached_benefits = json.load(f)
    if set(cached_benefits.keys()) == set(chunks_df["chunk_id"].tolist()):
        all_benefits = cached_benefits
        # Identify previously-failed chunks so we can re-run only those
        _err_df = du.read_csv_safe(benefits_errors_file)
        if not _err_df.empty:
            _failed_ids = set(_err_df["chunk_id"].tolist())
            retry_ids = {k for k in _failed_ids
                         if k in all_benefits and len(all_benefits[k]["benefits"]) == 0}
        if retry_ids:
            show_status(f"Re-extracting {len(retry_ids)} previously-failed chunks...")
        else:
            print(f"Using cached extractions for {len(all_benefits)} paragraphs")
    else:
        print("Cache mismatch — will re-extract all chunks")
        all_benefits = None

# Determine which chunks to run
if all_benefits is None:
    all_benefits = {}
    chunks_to_run = list(chunks_df.iterrows())
elif retry_ids:
    chunks_to_run = [(i, row) for i, row in chunks_df.iterrows()
                     if row["chunk_id"] in retry_ids]
else:
    chunks_to_run = []

if chunks_to_run:
    chunk_metadata = chunks_df.set_index("chunk_id")[["technology", "year", "source_file"]].to_dict("index")

    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {
            executor.submit(du.extract_phrases, row, "benefit", client): row[1]["chunk_id"]
            for row in chunks_to_run
        }
        for future in tqdm(as_completed(futures), total=len(futures), desc="Extracting benefits"):
            result = future.result()
            results.append(result)
            meta = chunk_metadata[result.chunk_id]
            all_benefits[result.chunk_id] = {
                "benefits": result.retained_phrases,
                "technology": meta["technology"],
                "year": int(meta["year"]) if pd.notna(meta["year"]) else None,
                "source_file": meta["source_file"],
            }

    with open(benefits_cache_file, "w") as f:
        json.dump(all_benefits, f, indent=2)

    du.write_extraction_diagnostics(results, "benefit", OUTPUT_FOLDER)
    failures = [r for r in results if r.error]
    failure_rate = len(failures) / max(1, len(chunks_to_run))
    if failure_rate > 0.05:
        show_warning(f"Benefit extraction failure rate ({failure_rate:.1%}) exceeds 5% — check extraction_errors_benefit.csv")
    elif failures:
        show_warning(f"{len(failures)} benefit extraction failures ({failure_rate:.1%}). See extraction_errors_benefit.csv")

    show_complete(f"Extracted/updated benefits for {len(chunks_to_run)} paragraphs")

# Flatten to individual benefit rows
benefit_rows = [
    {
        "chunk_id": chunk_id,
        "benefit": benefit,
        "technology": data["technology"],
        "year": data["year"],
        "source_file": data["source_file"],
    }
    for chunk_id, data in all_benefits.items()
    for benefit in data["benefits"]
]

_benefit_cols = ["chunk_id", "benefit", "technology", "year", "source_file"]
benefits_df = pd.DataFrame(benefit_rows, columns=_benefit_cols) if benefit_rows else pd.DataFrame(columns=_benefit_cols)
benefits_df["benefit_id"] = [f"benefit_{i}" for i in range(len(benefits_df))]

print(f"\nExtraction Summary:")
print(f"  Paragraphs in cache: {len(all_benefits)}")
print(f"  Total benefit phrases: {len(benefits_df)}")
print(f"  Avg benefits per paragraph: {len(benefits_df) / max(1, len(all_benefits)):.2f}")
print(f"  Paragraphs with no benefits: {sum(1 for d in all_benefits.values() if len(d['benefits']) == 0)}")

benefits_df.to_csv(OUTPUT_FOLDER / "extracted_benefits.csv", index=False)

# Retry any chunks that still failed after the extraction loop above.
# Runs in-process so all retry log lines appear in this cell's output.
_benefit_errors = OUTPUT_FOLDER / "extraction_errors_benefit.csv"
_has_benefit_errors = not du.read_csv_safe(_benefit_errors).empty
if _has_benefit_errors:
    import importlib.util as _ilu
    _spec = _ilu.spec_from_file_location(
        "retry_failed_extractions",
        _REPO_ROOT / "scripts" / "retry_failed_extractions.py",
    )
    _retry_mod = _ilu.module_from_spec(_spec)
    _spec.loader.exec_module(_retry_mod)
    _retry_mod._retry_kind("benefit", chunks_df, client, dry_run=False, max_workers=5)
    benefits_df = pd.read_csv(OUTPUT_FOLDER / "extracted_benefits.csv")
    print(f"Post-retry: {len(benefits_df)} total benefit rows")

This code segment ensures that the `benefits_df` DataFrame accurately reflects the technology associated with each benefit phrase by merging it with `chunks_df`, which contains relevant metadata. By prioritizing the 'technology' information from `chunks_df`, the analysis maintains consistency and completeness in the dataset, which is crucial for subsequent clustering and interpretation of benefits in the context of public dialogue. This step is essential for ensuring that the clustering results are meaningful and aligned with the technological themes present in the documents being analyzed.

In [ ]:
# Make sure benefits_df knows the technology of each paragraph (safe merge)

# Prefer chunks_df's 'technology' if 'technology_meta' doesn't exist there
tech_col = "technology_meta" if "technology_meta" in chunks_df.columns else "technology"

benefits_df = benefits_df.merge(
    chunks_df[["chunk_id", tech_col]].rename(columns={tech_col: "technology_meta"}),
    on="chunk_id",
    how="left"
)

# If benefits_df already has 'technology' and it's missing where technology_meta exists, fill it
if "technology" in benefits_df.columns:
    benefits_df["technology"] = benefits_df["technology"].fillna(benefits_df["technology_meta"])
else:
    benefits_df["technology"] = benefits_df["technology_meta"]

This cell inspects and displays a sample of benefit phrases extracted from the dataset, ensuring that the language used is not overly specific to particular technologies. By confirming the removal of technology-specific terminology, it helps validate the generalizability of the extracted benefits, which is crucial for subsequent analysis and clustering of concerns across diverse public dialogues. This step is essential for maintaining the integrity of the semantic embeddings and ensuring that the clustering process captures broader themes rather than niche, technology-specific insights.

In [ ]:
# @title Inspect sample benefit extractions

import pandas as pd

print("Sample extracted benefits by technology:")
print("(Checking that technology-specific language has been removed)\n")

# Identify the text column in benefits_df
candidate_cols = ["benefit", "benefit_phrase", "phrase", "extracted_phrase", "text"]
phrase_col = next((c for c in candidate_cols if c in benefits_df.columns), None)

if phrase_col is None:
    raise KeyError(
        f"Couldn't find a benefit text column in benefits_df. "
        f"Looked for {candidate_cols}. Available columns: {list(benefits_df.columns)}"
    )

# Identify technology column (you appear to have technology_meta)
tech_col = "technology_meta" if "technology_meta" in benefits_df.columns else (
    "technology" if "technology" in benefits_df.columns else None
)

if tech_col is None:
    raise KeyError(
        f"Couldn't find a technology column in benefits_df. "
        f"Available columns: {list(benefits_df.columns)}"
    )

for tech in benefits_df[tech_col].dropna().unique()[:6]:
    tech_benefits = benefits_df[benefits_df[tech_col] == tech][phrase_col].head(8).tolist()
    print(f"\n{tech}:")
    for b in tech_benefits:
        print(f"  • {b}")

This cell generates semantic embeddings for benefit phrases extracted from public dialogue documents, leveraging a large language model (LLM) to capture the contextual meaning of each phrase. By caching the embeddings and their corresponding IDs, it optimizes the process for future analyses while ensuring data consistency. This step is crucial for the subsequent clustering analysis, as it transforms qualitative text data into a quantitative format that facilitates the identification of patterns and relationships among the benefits articulated in the public discourse.

In [ ]:
# @title Generate semantic embeddings for benefit phrases

embeddings_file = CHECKPOINT_FOLDER / "benefit_embeddings.npy"
benefit_ids_file = CHECKPOINT_FOLDER / "benefit_ids.json"

if embeddings_file.exists() and benefit_ids_file.exists():
    print("Found cached embeddings...")
    benefit_embeddings = np.load(embeddings_file)
    with open(benefit_ids_file) as f:
        cached_benefit_ids = json.load(f)
    if cached_benefit_ids == benefits_df["benefit_id"].tolist():
        print(f"Using cached embeddings: {benefit_embeddings.shape}")
    else:
        print("Cache mismatch - regenerating")
        benefit_embeddings = None
else:
    benefit_embeddings = None

if benefit_embeddings is None:
    show_status(f"Generating embeddings for {len(benefits_df)} benefit phrases...")
    texts = benefits_df["benefit"].tolist()
    all_embeddings = []
    failed_batch_starts = []

    for i in tqdm(range(0, len(texts), EMBEDDING_BATCH_SIZE), desc="Embedding"):
        batch = texts[i:i + EMBEDDING_BATCH_SIZE]
        try:
            all_embeddings.append(du.get_embeddings_batch(batch, client))
        except Exception as e:
            print(f"Error on batch {i}: {e}")
            failed_batch_starts.append(i)
        time.sleep(0.1)

    if failed_batch_starts:
        raise RuntimeError(
            f"{len(failed_batch_starts)} benefit embedding batch(es) failed: "
            f"starting indices {failed_batch_starts}. Re-run to regenerate."
        )

    benefit_embeddings = np.vstack(all_embeddings) if all_embeddings else np.empty((0, 0))
    np.save(embeddings_file, benefit_embeddings)
    with open(benefit_ids_file, "w") as f:
        json.dump(benefits_df["benefit_id"].tolist(), f)
    show_complete(f"Generated embeddings: {benefit_embeddings.shape}")

assert benefit_embeddings.shape[0] == len(benefits_df), (
    f"Benefit embeddings count ({benefit_embeddings.shape[0]}) does not match "
    f"benefits_df rows ({len(benefits_df)}). Delete {embeddings_file} and re-run."
)
if benefit_embeddings.size:
    print(f"Embedding dimensions: {benefit_embeddings.shape[1]}")
else:
    print("No embeddings generated (benefits_df empty).")

This cell serves as a verification step to ensure that all necessary output files generated during the extraction phase are saved correctly before proceeding to the clustering analysis. By checking the existence and size of key artifacts, such as extracted concern and benefit phrases along with their embeddings, it helps prevent errors in subsequent steps of the pipeline, ensuring a smooth transition to the clustering phase and maintaining the integrity of the overall analysis.

In [ ]:
# @title Extraction checkpoint — confirm artifacts are saved
# Run this at the end of 01_processing to verify that all raw artifacts
# have been written before starting 01a_clustering.
from pathlib import Path as _Path
_out  = OUTPUT_FOLDER
_ckpt = CHECKPOINT_FOLDER

_EXPECTED = {
    _out  / "paragraph_chunks.csv":     "chunks",
    _out  / "extracted_concerns.csv":   "concern phrases",
    _out  / "extracted_benefits.csv":   "benefit phrases",
    _ckpt / "concern_embeddings.npy":   "concern embeddings",
    _ckpt / "benefit_embeddings.npy":   "benefit embeddings",
}

_ok = True
for _p, _label in _EXPECTED.items():
    _sz = f"{_p.stat().st_size / 1e6:.1f} MB" if _p.exists() else "MISSING"
    _flag = "OK  " if _p.exists() else "MISS"
    print(f"  {_flag}  {_label:<25}  {_sz}")
    if not _p.exists():
        _ok = False

if not _ok:
    raise RuntimeError("Some extraction artifacts are missing — check earlier cells.")
print("\nAll extraction artifacts present. Run 01a_clustering.ipynb next.")